# Setup

In [1]:
# Setup
import json
import os
import sys
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv, set_key
import random
from pathlib import Path

# hashing for signing
import hashlib
import hmac

# requests
import requests

# data munging
import pandas as pd
import numpy as np

# helper functions
from tiktok_api_helpers import *

## API Setup

In [2]:
new_access_token = False
use_refresh_token = False

In [3]:
if (new_access_token | use_refresh_token):

    # get new acces_token
    token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

    if use_refresh_token:
        auth_code = os.environ.get("TIKTOK_REFRESH_TOKEN")
        
    params = {
        "app_key": app_key,
        "app_secret": app_secret,
        "auth_code": auth_code,
        "grant_type": "authorized_code",
    }
    
    response = requests.get(token_url, params=params, timeout=15)
    data = response.json()['data']
    access_token = data['access_token']
    print(data)

    # save tokens in .env file
    set_key(".env", "TIKTOK_ACCESS_TOKEN", data['access_token'])
    set_key(".env", "TIKTOK_REFRESH_TOKEN", data['refresh_token'])

else:
    access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
    refresh_token = os.environ.get("TIKTOK_REFRESH_TOKEN")

# Step 1. Monitor Sample Application Status

PENDING, AWAITING_SHIPMENT, SHIPPED, CONTENT_PENDING, REJECT_CANCELLED, OVERDUE_CANCELLED, UNFULFILL_CANCELLED, DEL_OPEN_COLLAB, SELLER_NOT_SHIP_CANCELLED, WITHDRAW_CANCELLED, UNFULFILLABLE_CANCELLED, OPS_CANCELLED, OPS_FAILED, OPS_COMPLETED, COMPLETED

In [4]:
statuses = [
    "AWAITING_SHIPMENT", "SHIPPED", "CONTENT_PENDING", 
    "OPS_COMPLETED", "COMPLETED"
]

all_sample_applications = []
for status in statuses:
    print(f"Fetching status: {status}")
    results = search_all_sample_applications(status=status)
    all_sample_applications.extend(results)

print(f"\nTotal combined results: {len(all_sample_applications)}")

Fetching status: AWAITING_SHIPMENT
Page 1: 11 application(s) (total so far: 11)
Fetching status: SHIPPED
Page 1: 50 application(s) (total so far: 50)
Page 2: 17 application(s) (total so far: 67)
Fetching status: CONTENT_PENDING
Page 1: 50 application(s) (total so far: 50)
Page 2: 23 application(s) (total so far: 73)
Fetching status: OPS_COMPLETED
Page 1: 0 application(s) (total so far: 0)
Fetching status: COMPLETED
Page 1: 44 application(s) (total so far: 44)

Total combined results: 195


In [5]:
df_all_sample_applications = pd.json_normalize(all_sample_applications)

In [6]:
df_all_sample_applications.to_csv(SAMPLE_APPLICATIONS_CSV, index=False)

In [7]:
df_all_sample_applications['status'].value_counts()

status
CONTENT_PENDING      73
SHIPPED              67
COMPLETED            44
AWAITING_SHIPMENT    11
Name: count, dtype: int64

# Step 2. Create Conversations

## Check existing conversations

In [8]:
ALL_CONVERSATIONS_CSV = "messaging/all_conversations.csv"
CONVERSATIONS_MANIFEST_CSV = "messaging/creators_conversations_manifest.csv"

In [9]:
all_conversations = []
page_token = ""
page_num = 1

while True:
    result = get_conversation_list(
        page_size=50,
        page_token=page_token,
        conversation_status="ALL",
        only_need_conversation_id=False,
    )

    if result.get("code") != 0:
        print(f"  ⚠️  Page {page_num} failed, stopping here: {result}")
        break

    data = result.get("data", {}) or {}
    conversations = data.get("conversations", [])
    all_conversations.extend(conversations)

    print(f"Page {page_num}: {len(conversations)} conversation(s) (total so far: {len(all_conversations)})")

    if not data.get("has_more"):
        break

    page_token = data.get("next_page_token", "")
    if not page_token:
        break

    page_num += 1
    time.sleep(0.5)

print(f"\nDone. {len(all_conversations)} total conversation(s) collected.")

Page 1: 50 conversation(s) (total so far: 50)
Page 2: 50 conversation(s) (total so far: 100)
Page 3: 50 conversation(s) (total so far: 150)
Page 4: 50 conversation(s) (total so far: 200)
Page 5: 50 conversation(s) (total so far: 250)
Page 6: 50 conversation(s) (total so far: 300)
Page 7: 50 conversation(s) (total so far: 350)
Page 8: 50 conversation(s) (total so far: 400)
Page 9: 50 conversation(s) (total so far: 450)
Page 10: 50 conversation(s) (total so far: 500)
Page 11: 50 conversation(s) (total so far: 550)
Page 12: 50 conversation(s) (total so far: 600)
Page 13: 50 conversation(s) (total so far: 650)
Page 14: 50 conversation(s) (total so far: 700)
Page 15: 50 conversation(s) (total so far: 750)
Page 16: 50 conversation(s) (total so far: 800)
Page 17: 50 conversation(s) (total so far: 850)
Page 18: 50 conversation(s) (total so far: 900)
Page 19: 50 conversation(s) (total so far: 950)
Page 20: 50 conversation(s) (total so far: 1000)
Page 21: 50 conversation(s) (total so far: 1050)


In [10]:
df_all_conversations = pd.DataFrame(all_conversations)

In [11]:
df_all_conversations.to_csv(ALL_CONVERSATIONS_CSV, index=False)

## Track conversation IDs of all creators

In [213]:
# load all creators
df_creators_list = pd.read_excel("all_creators_sorted.xlsx", sheet_name="LIST_CREATOR", usecols=[1, 2, 25])
df_creators = pd.read_csv(CONSOLIDATED_CSV)
df_target_collab = pd.read_csv(TARGET_COLLAB_CREATORS_CSV)
df_target_collab['target_collaboration_id'] = df_target_collab['target_collaboration_id'].astype(str)

In [13]:
# merge all extracted data
df_creators_messaging = df_creators_list \
    .merge(df_creators[['username', 'creator_open_id']], how='left', on="username") \
    .merge(df_target_collab.rename(columns={'name':'target_collaboration_name'}).drop(['batch_name'], axis=1), how='left', on="creator_open_id") \
    .merge(df_all_sample_applications[['creator.creator_open_id', 'status']].rename(columns={'creator.creator_open_id':'creator_open_id', 'status':'sample_status'}), how='left', on="creator_open_id") \
    .merge(df_all_conversations[['creator_im_id', 'id', 'username']].rename(columns={'id':'conversation_id'}), how='left', on="username")

## [Optional] Create Conversation if invited to collab

In [50]:
list_create_conversation = df_creators_messaging.loc[df_creators_messaging['target_collaboration_id'].notnull() & df_creators_messaging['conversation_id'].isnull(), 'creator_open_id'].tolist()
# list_create_conversation = df_creators_messaging.loc[df_creators_messaging['sample_status'].notnull() & df_creators_messaging['conversation_id'].isnull(), 'creator_open_id'].tolist()
print(f"Generating conversation IDs for {len(list_create_conversation)} new creators.")

Generating conversation IDs for 5 new creators.


In [51]:
conversation_rows = []
failed_conversations = []

In [52]:
for i, creator_open_id in enumerate(list_create_conversation, start=1):
    result = create_conversation_with_creator(creator_open_id, only_need_conversation_id=False)

    if result.get("code") != 0:
        print(f"  ⚠️  Failed: {result}")
        failed_conversations.append(creator_open_id)
    else:
        data = result["data"]
        conversation_rows.append({
            "creator_open_id": creator_open_id,
            "conversation_id": data.get("conversation_id"),
            "creator_im_id": data.get("creator_im_id"),
        })
        print(f"  ✅ {creator_open_id} -> conversation_id: {data.get('conversation_id')}")

    time.sleep(0.5)

print(f"\nDone. {len(conversation_rows)} succeeded, {len(failed_conversations)} failed.")

  ✅ CICFKwAAAACOV4AEeERJl-yievH_HBssqkU7H--75ImCOWkPzQmnsg -> conversation_id: 7659390859917492500
  ✅ QnrT-QAAAACOV4AEeERJl-yievH_HBssaAq-MTUAOiLzfx_rdNpAkg -> conversation_id: 7658276473475924242
  ✅ ctTJ0QAAAACOV4AEeERJl-yievH_HBssp8B68npzmhu7eLphjZP09Q -> conversation_id: 7657528465915248903
  ✅ 5SJoxAAAAACOV4AEeERJl-yievH_HBssnksHd5GVqVNRX-pZpekVRw -> conversation_id: 7658216864620134676
  ✅ JYHREwAAAACOV4AEeERJl-yievH_HBsscEhMRcfgGP6DJPzpaMdC2w -> conversation_id: 7658238340437770516

Done. 5 succeeded, 0 failed.


In [53]:
if conversation_rows:
    df_conversations_new = pd.DataFrame(conversation_rows)
    file_exists = Path(ALL_CONVERSATIONS_NEW_CSV).exists()
    df_conversations_new.to_csv(ALL_CONVERSATIONS_NEW_CSV, mode="a", header=not file_exists, index=False)

    print(f"\nAppended {len(df_conversations_new)} row(s) to {TARGET_COLLAB_CREATORS_CSV}")


Appended 5 row(s) to collabs/target_collaboration_creators.csv


In [54]:
df_conversations_new = pd.read_csv(ALL_CONVERSATIONS_NEW_CSV)

In [55]:
df_all_conversations = pd.concat([df_all_conversations[['creator_im_id', 'conversation_id', 'username']],
           df_conversations_new.merge(df_creators[['username', 'creator_open_id']], how='left', on="creator_open_id")[['creator_im_id', 'conversation_id', 'username']]]).drop_duplicates(subset=['username']).reset_index(drop=True)

In [56]:
df_all_conversations.to_csv(ALL_CONVERSATIONS_CSV, index=False)

# Step 3. Bulk Send Message

In [246]:
# merge all extracted data
df_creators_messaging = df_creators_list \
    .merge(df_creators[['username', 'creator_open_id']], how='left', on="username") \
    .merge(df_target_collab.rename(columns={'name':'target_collaboration_name'}).drop(['batch_name'], axis=1), how='left', on="creator_open_id") \
    .merge(df_all_sample_applications[['creator.creator_open_id', 'status']].rename(columns={'creator.creator_open_id':'creator_open_id', 'status':'sample_status'}), how='left', on="creator_open_id") \
    .merge(df_all_conversations[['creator_im_id', 'conversation_id', 'username']], how='left', on="username")

# change all conversation_id to str
df_creators_messaging['conversation_id'] = df_creators_messaging['conversation_id'].apply(lambda x: str(x) if pd.notnull(x) else np.nan)

In [247]:
dict_conv_to_username = df_creators_messaging.dropna(subset=['conversation_id']).set_index('conversation_id')['username'].to_dict()
dict_conv_to_collab =  df_creators_messaging.dropna(subset=['conversation_id']).set_index('conversation_id')['target_collaboration_id'].to_dict()

## Track Message Status based on manifest

In [248]:
df_collab_invite = pd.read_csv(COLLAB_INVITE_MANIFEST_CSV)
df_viber_invite = pd.read_csv(VIBER_INVITE_MANIFEST_CSV)

In [249]:
df_creators_messaging['invited_to_join_collab'] =  df_creators_messaging['conversation_id'].isin(df_collab_invite['conversation_id'].astype(str))
df_creators_messaging['invited_to_viber_grp'] =  df_creators_messaging['conversation_id'].isin(df_viber_invite['conversation_id'].astype(str))

## Send Viber Group invite

In [241]:
list_viber_invite = df_creators_messaging.loc[df_creators_messaging['sample_status'].notnull() & ~df_creators_messaging['invited_to_viber_grp'], 'conversation_id'].astype(str).tolist()
print(f"Sending Viber invites for {len(list_viber_invite)} new creators.")

Sending Viber invites for 0 new creators.


In [188]:
message_thank_you = "Hi {}! Thank you so much for accepting our invite, super excited to work with you! Sending over a photo with QR codes para sa content brief and Viber community namin. Just scan para ma access mo!\n\n Feel free to message me here anytime kung may mga tanong ka. Looking forward to creating with you!"

message_viber_qrcode = {
    'url': 'https://p16-oec-sg.ibyteimg.com/tos-alisg-i-aphluv4xwc-sg/de85540122b94ac9bf6c801faf19d01c~tplv-aphluv4xwc-origin-image.image?dr=15570&t=555f072d&ps=933b5bde&shp=5566cfe3&shcp=3c3d9ffb&idc=my&from=1432801251',
    'width': 1280,
    'height': 905
}

In [189]:
# Skip conversations already FULLY sent (both TEXT and IMAGE succeeded) in a previous run — a partial failure (e.g. text sent but image failed) should
# still be retried, not silently treated as done.
if Path(VIBER_INVITE_MANIFEST_CSV).exists():
    prior = pd.read_csv(VIBER_INVITE_MANIFEST_CSV, dtype={"conversation_id": str})
    already_sent = set(prior.loc[prior["text_sent"] & prior["image_sent"], "conversation_id"])
else:
    already_sent = set()

to_send = [c for c in list_viber_invite if c not in already_sent]
skipped = len(list_viber_invite) - len(to_send)
if skipped:
    print(f"Skipping {skipped} conversation(s) already sent in a previous run.")

In [190]:
results = []

for i, conversation_id in enumerate(to_send, start=1):
    print(f"{i}/{len(to_send)}: messaging {conversation_id}...")

    username = dict_conv_to_username[conversation_id]
    text_result = send_im_message(conversation_id, "TEXT", {"content": message_thank_you.format(username)})
    text_ok = text_result.get("code") == 0
    if not text_ok:
        print(f"  ⚠️  TEXT failed: {text_result}")
    time.sleep(DELAY_BETWEEN_PAGES)

    image_result = send_im_message(conversation_id, "IMAGE", message_viber_qrcode)
    image_ok = image_result.get("code") == 0
    if not image_ok:
        print(f"  ⚠️  IMAGE failed: {image_result}")
    time.sleep(DELAY_BETWEEN_PAGES)

    if text_ok and image_ok:
        print("  ✅ Both messages sent")

    results.append({
        "conversation_id": conversation_id,
        "text_sent": text_ok,
        "image_sent": image_ok,
    })

    # Save progress after every creator, not just at the end — so a crash
    # partway through doesn't lose track of who's already been messaged.
    file_exists = Path(VIBER_INVITE_MANIFEST_CSV).exists()
    pd.DataFrame([results[-1]]).to_csv(VIBER_INVITE_MANIFEST_CSV, mode="a", header=not file_exists, index=False)

fully_sent = sum(1 for r in results if r["text_sent"] and r["image_sent"])
print(f"\nDone. {fully_sent}/{len(results)} creator(s) got both messages successfully this run.")

1/92: messaging 7662759577196314888...
  ✅ Both messages sent
2/92: messaging 7663053189420384532...
  ✅ Both messages sent
3/92: messaging 7663053673941795092...
  ✅ Both messages sent
4/92: messaging 7662761695468077330...
  ✅ Both messages sent
5/92: messaging 7663054331399487764...
  ✅ Both messages sent
6/92: messaging 7658276473475924242...
  ✅ Both messages sent
7/92: messaging 7663054705639260424...
  ✅ Both messages sent
8/92: messaging 7663055642948813064...
  ✅ Both messages sent
9/92: messaging 7657528465915248903...
  ✅ Both messages sent
10/92: messaging 7663055770891976978...
  ✅ Both messages sent
11/92: messaging 7663055878425674004...
  ✅ Both messages sent
12/92: messaging 7663056158630314248...
  ✅ Both messages sent
13/92: messaging 7662759932811084050...
  ✅ Both messages sent
14/92: messaging 7663056228100833543...
  ✅ Both messages sent
15/92: messaging 7663056249831653650...
  ✅ Both messages sent
16/92: messaging 7663056334040596752...
  ✅ Both messages sent
1

## Send Target Collab Invite

In [252]:
list_collab_invite = df_creators_messaging.loc[~df_creators_messaging['invited_to_join_collab'] & df_creators_messaging['target_collaboration_id'].notnull(), 'conversation_id'].astype(str).tolist()
print(f"Sending collab invites for {len(list_collab_invite)} new creators.")

Sending collab invites for 4529 new creators.


In [253]:
message_invite = "Hi {}! 😊\n\nI'm Coleen, founder of Vitami. Familiar ka ba sa PMS? Yung cramps, intense mood swings, bloating, at acne na kasabay ng period? Whether ikaw mismo nakakaramdam nun or may kakilala kang nageexperience nito, we all know how rough that week can get. Kaya gumawa ako ng product specifically for that kasi hindi dapat normal ang magtiis every month. My goal is simple: make period week lighter for every woman.\n\nWould you be open to creating content with us and helping other girls out? You'll get 20% commission per sale plus a free sample to try!\n\n💚 Coleen"

In [254]:
# Skip conversations already FULLY sent (both TEXT and TARGET_COLLAB_CARD succeeded) in a previous run — a partial failure (e.g. text sent but image failed) should
# still be retried, not silently treated as done.
if Path(COLLAB_INVITE_MANIFEST_CSV).exists():
    prior = pd.read_csv(COLLAB_INVITE_MANIFEST_CSV, dtype={"conversation_id": str})
    already_sent = set(prior.loc[prior["text_sent"] & prior["collab_sent"], "conversation_id"])
else:
    already_sent = set()

to_send = [c for c in list_collab_invite if c not in already_sent]
skipped = len(list_collab_invite) - len(to_send)
if skipped:
    print(f"Skipping {skipped} conversation(s) already sent in a previous run.")

In [ ]:
results = []

for i, conversation_id in enumerate(to_send, start=1):
    print(f"{i}/{len(to_send)}: messaging {conversation_id}...")

    username = dict_conv_to_username[conversation_id]
    target_collab_id = dict_conv_to_collab[conversation_id]
    
    text_result = send_im_message(conversation_id, "TEXT", {"content": message_invite.format(username)})
    text_ok = text_result.get("code") == 0
    if not text_ok:
        print(f"  ⚠️  TEXT failed: {text_result}")
    time.sleep(DELAY_BETWEEN_PAGES)

    collab_result = send_im_message(conversation_id, "TARGET_COLLABORATION_CARD", {"target_collaboration_id": target_collab_id})
    collab_ok = collab_result.get("code") == 0
    if not collab_ok:
        print(f"  ⚠️  TARGET_COLLAB failed: {collab_result}")
    time.sleep(DELAY_BETWEEN_PAGES)

    if text_ok and collab_ok:
        print("  ✅ Both messages sent")

    results.append({
        "conversation_id": conversation_id,
        "text_sent": text_ok,
        "collab_sent": collab_ok,
    })

    # Save progress after every creator, not just at the end — so a crash
    # partway through doesn't lose track of who's already been messaged.
    file_exists = Path(COLLAB_INVITE_MANIFEST_CSV).exists()
    pd.DataFrame([results[-1]]).to_csv(COLLAB_INVITE_MANIFEST_CSV, mode="a", header=not file_exists, index=False)

fully_sent = sum(1 for r in results if r["text_sent"] and r["collab_sent"])
print(f"\nDone. {fully_sent}/{len(results)} creator(s) got both messages successfully this run.")

1/4529: messaging 7660854389301444885...
  ✅ Both messages sent
2/4529: messaging 7663050627153215764...
  ✅ Both messages sent
3/4529: messaging 7663050636116328722...
  ✅ Both messages sent
4/4529: messaging 7663050632018821383...
  ✅ Both messages sent
5/4529: messaging 7660854310464815381...
  ✅ Both messages sent
6/4529: messaging 7663050638625276181...
  ✅ Both messages sent
7/4529: messaging 7660854542264008981...
  ✅ Both messages sent
8/4529: messaging 7660854414731526420...
  ✅ Both messages sent
9/4529: messaging 7660854237493084437...
  ✅ Both messages sent
10/4529: messaging 7663050645633597716...
  ✅ Both messages sent
11/4529: messaging 7660854228437467412...
  ✅ Both messages sent
12/4529: messaging 7660854315342610709...
  ✅ Both messages sent
13/4529: messaging 7663050674233557249...
  ✅ Both messages sent
14/4529: messaging 7663050655892439304...
  ✅ Both messages sent
15/4529: messaging 7660854362847314196...
  ✅ Both messages sent
16/4529: messaging 766305068423695